# Dynamic Programming — manutenção de escadas rolantes

**Turma: 2ESPS**

- Gabriel Cutrim — RM 568059
- Guilherme Pereira — RM 567487
- Gustavo Firmino Barbosa — RM 566903

Este notebook contém somente a entrega de Dynamic Programming: recorrências,
ranking, grafo, BFS, Dijkstra, A* e plano de manutenção.

**Dados reais:** o arquivo original `ecr_09-26.XLSX` tem 104.119 linhas e 22 colunas.
`dados/historico_avarias.csv` preserva as mesmas linhas, sete campos usados pelos
algoritmos e uma assinatura técnica para manter a deduplicação original sem
publicar os textos livres. A procedência e os hashes estão em `dados/procedencia_historico.json`.
O histórico não contém sensores físicos; por isso, o índice da primeira parte
acumula durações de avarias e não representa degradação física.

**Complemento de sensores SIMULADOS:** equipamentos fictícios `SIM-`, parâmetros
didáticos e resultados separados das observações reais. Sua aceitação como
alternativa ao dataset de sensores depende do professor. Nada nesta simulação
é apresentado como medição da Motiva.


## 1. Definição do indicador e recorte
      **Adaptação explícita:** D(n) acumula minutos registrados em avarias marcadas
      `Indisponível = X`, por equipamento individual, no último ano observado.
      D(0)=0 e D(n)=D(n−1)+duração(n). É um indicador de carga histórica de avarias,
      **não degradação física nem probabilidade de falha**. Intervalos sobrepostos
      podem contar duas vezes; somas não representam disponibilidade cronológica.
      Durações ausentes não viram zero: são auditadas, não contribuem e a cobertura
      é exibida. Tempos fora da janela anual são recortados. Serviços de 45 min são
      hipóteses de planejamento, não estimativas extraídas da base.

In [1]:
from pathlib import Path
import sys, json
ROOT = Path.cwd().resolve()
if not (ROOT / 'preparacao.py').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from preparacao import carregar, preparar
plt.rcParams.update({'figure.figsize': (10, 4), 'axes.grid': True})
bruto, fonte = carregar()
df, auditoria = preparar(bruto)
OUT = ROOT / 'resultados'
OUT.mkdir(exist_ok=True)
print('Fonte:', fonte.name)
print('Dimensões originais:', bruto.shape)
display(pd.Series(auditoria, name='contagem').to_frame())

from algoritmos import recursiva, memoizada, tabulacao, montar_grafo, bfs, dijkstra, astar, reconstruir, criar_planejador
from timeit import repeat
import math

Fonte: historico_avarias.csv
Dimensões originais: (104119, 8)


,contagem
linhas_originais,104119
duplicatas_exatas,142
linhas_preparadas,103977
inicio_ausente,25517
fim_ausente,27089
duracao_negativa,1
data_futura,1
duracao_valida,76652
equipamento_ausente,0


In [2]:
ano = int(df.ano.dropna().max())
inicio_janela, fim_janela = pd.Timestamp(ano, 1, 1), pd.Timestamp(ano + 1, 1, 1)
recorte = df[df.equipamento_identificado & df.ano.eq(ano).fillna(False)
             & df['Indisponível'].eq('X')].copy()
recorte['incremento'] = (recorte.fim.clip(upper=fim_janela) - recorte.inicio).dt.total_seconds()/60
recorte['incremento'] = recorte.incremento.where(recorte.duracao_min.notna())
recorte = recorte.sort_values(['escada_id', 'inicio', 'Nota'], kind='stable')
sequencias, linhas = {}, []
for escada, grupo in recorte.groupby('escada_id'):
    valores = grupo.incremento.dropna().tolist()
    sequencias[escada] = valores
    linhas.append(dict(escada_id=escada, estacao=grupo.codigo_estacao.iloc[0],
        criticidade=tabulacao(valores)[-1] if valores else np.nan,
        registros=len(grupo), validos=len(valores), cobertura=len(valores)/len(grupo), servico_min=45))
ranking = pd.DataFrame(linhas).sort_values(['criticidade', 'escada_id'], ascending=[False, True])
display(ranking.head(20))
ranking.to_csv(OUT / 'ranking_criticidade.csv', index=False)
display(Markdown(f"Janela: ano **{ano}**, {len(recorte)} registros e {len(ranking)} equipamentos. "
    f"O maior índice é de **{ranking.iloc[0].escada_id}**, com {ranking.iloc[0].criticidade:.2f} minutos somados. "
    f"Existem {recorte.incremento.isna().sum()} contribuições ausentes. O ranking prioriza soma observada, "
    "devendo ser revisado quando a cobertura é baixa; não inclui equipamentos sem registros na janela."))

,escada_id,estacao,criticidade,registros,validos,cobertura,servico_min
133,SP04-6-ECR-PIN-ER21,PIN,138073.166667,30,30,1.0,45
138,SP04-6-ECR-PIN-ER26,PIN,91417.883333,6,6,1.0,45
127,SP04-6-ECR-PIN-ER15,PIN,30370.600000,15,15,1.0,45
159,SP04-6-ECR-PTA-ER15,PTA,28487.533333,72,72,1.0,45
134,SP04-6-ECR-PIN-ER22,PIN,23576.883333,46,46,1.0,45
46,SP04-6-ECR-FRE-ER13,FRE,22870.150000,90,90,1.0,45
22,SP04-6-ECR-FAL-ER09,FAL,18185.383333,16,16,1.0,45
161,SP04-6-ECR-PTA-ER17,PTA,13512.966667,83,83,1.0,45
132,SP04-6-ECR-PIN-ER20,PIN,13159.283333,17,17,1.0,45
6,SP04-6-ECR-BUT-ER07,BUT,12870.683333,24,24,1.0,45


Janela: ano **2026**, 5520 registros e 196 equipamentos. O maior índice é de **SP04-6-ECR-PIN-ER21**, com 138073.17 minutos somados. Existem 1 contribuições ausentes. O ranking prioriza soma observada, devendo ser revisado quando a cobertura é baixa; não inclui equipamentos sem registros na janela.

## 2. Três versões e desempenho
       Todas as versões são O(n) em uma consulta: esta recorrência tem somente
       uma chamada por passo, sem explosão exponencial. A memoização acrescenta
       custo de dicionário, mas reutiliza prefixos. Consultar todos os prefixos
       separadamente custa O(n²) na pura e O(n) com cache compartilhado/tabulação.
       A recursão depende do limite de pilha: sequências longas são somadas em
       blocos de até 300 eventos nas duas versões recursivas, sem elevar o limite.
       A comparação direta mede prefixos até 300; o ranking integral usa tabulação.

In [3]:
def por_blocos(funcao, valores, tamanho=300):
    return sum(funcao(valores[i:i+tamanho]) for i in range(0, len(valores), tamanho))
verificacoes = []
for escada, valores in sequencias.items():
    a = por_blocos(recursiva, valores)
    b = por_blocos(memoizada, valores)
    c = tabulacao(valores)[-1]
    assert math.isclose(a, c, rel_tol=1e-10, abs_tol=1e-8)
    assert math.isclose(b, c, rel_tol=1e-10, abs_tol=1e-8)
    verificacoes.append(dict(escada_id=escada, recursiva=a, memoizada=b, tabulacao=c))
display(pd.DataFrame(verificacoes).head())
pd.DataFrame(verificacoes).to_csv(OUT / 'equivalencia_versoes.csv', index=False)
valores = max(sequencias.values(), key=len)[:300]
medicoes = []
for n in sorted(set([min(10,len(valores)), min(100,len(valores)), len(valores)])):
    amostra = valores[:n]
    for nome, func in [('recursiva', recursiva), ('memoizada_cache_frio', memoizada),
                       ('tabulacao', lambda x: tabulacao(x)[-1])]:
        tempos = repeat(lambda: func(amostra), number=100, repeat=7)
        medicoes.append(dict(n=n, versao=nome, mediana_us=np.median(tempos)*1e6/100,
                             minimo_us=min(tempos)*1e6/100))
desempenho = pd.DataFrame(medicoes); display(desempenho)
desempenho.to_csv(OUT / 'desempenho_recorrencias.csv', index=False)
cache = {0: 0.0}
prefixos_memo = [memoizada(valores, n, cache) for n in range(len(valores)+1)]
assert np.allclose(prefixos_memo, tabulacao(valores))
vencedora = desempenho[desempenho.n.eq(len(valores))].sort_values('mediana_us').iloc[0]
display(Markdown(f"Todas as {len(verificacoes)} sequências produziram totais equivalentes. "
    f"No maior prefixo medido, a menor mediana foi de **{vencedora.versao}** "
    f"({vencedora.mediana_us:.2f} µs). Tempos dependem do ambiente e de constantes de implementação. "
    "Os ensaios usam sete repetições de cem chamadas; cada chamada memoizada começa com cache vazio. "
    "A consulta de prefixos com cache compartilhado também foi validada."))

,escada_id,recursiva,memoizada,tabulacao
0,SP04-6-ECR-BUT-ER01,1829.983333,1829.983333,1829.983333
1,SP04-6-ECR-BUT-ER02,623.483333,623.483333,623.483333
2,SP04-6-ECR-BUT-ER03,1464.333333,1464.333333,1464.333333
3,SP04-6-ECR-BUT-ER04,2810.500000,2810.500000,2810.500000
4,SP04-6-ECR-BUT-ER05,3871.933333,3871.933333,3871.933333


,n,versao,mediana_us,minimo_us
0,10,recursiva,2.349000,1.298999
1,10,memoizada_cache_frio,2.522003,2.408000
2,10,tabulacao,0.649001,0.646999
3,100,recursiva,12.414004,11.239997
4,100,memoizada_cache_frio,27.815998,27.480000
5,100,tabulacao,5.836999,5.824002
6,105,recursiva,17.989003,17.066998
7,105,memoizada_cache_frio,29.565003,29.277001
8,105,tabulacao,6.123004,6.098002


Todas as 196 sequências produziram totais equivalentes. No maior prefixo medido, a menor mediana foi de **tabulacao** (6.12 µs). Tempos dependem do ambiente e de constantes de implementação. Os ensaios usam sete repetições de cem chamadas; cada chamada memoizada começa com cache vazio. A consulta de prefixos com cache compartilhado também foi validada.

## 3. Grafo e caminhos
       11 estações da Linha 4, um vértice adicional para TVS e uma base didática.
       Nomes das estações: [Motiva](https://trilhos.motiva.com.br/linha-4-amarela/estacoes/).
       Pesos em minutos, coordenadas esquemáticas, acesso viário e posição da base
       são hipóteses. O código TVS é tratado como terminal junto a Vila Sônia e
       precisa ser validado com o dicionário da empresa. A aresta BASE–PIN representa
       uma alternativa viária hipotética, permitindo distinguir menos trechos de
       menor tempo. Não se afirma que equipes utilizem essa rota na operação real.

In [4]:
rede = json.loads((ROOT / 'dados/rede_estacoes.json').read_text(encoding='utf-8'))
coords = {v['id']: (v['x'], v['y']) for v in rede['vertices']}
grafo = montar_grafo(coords, [(a['origem'], a['destino'], a['minutos']) for a in rede['arestas']])
display(pd.DataFrame([{'vertice': v, 'vizinhos_minutos': viz} for v, viz in grafo.items()]))
cadastro = df[df.equipamento_identificado][['escada_id', 'codigo_estacao']].drop_duplicates()
cadastro = cadastro.rename(columns={'codigo_estacao': 'estacao'})
assert not cadastro.escada_id.duplicated().any(), 'Escada vinculada a mais de uma estação'
faltantes = set(cadastro.estacao.dropna()) - set(grafo)
assert not faltantes, f'Estações sem vínculo: {faltantes}'
cadastro.to_csv(ROOT / 'dados/escadas_estacoes.csv', index=False)
print('Equipamentos individuais vinculados em toda a base:', len(cadastro))
trechos, pais_bfs = bfs(grafo, 'BASE')
dist, pais_dij, _ = dijkstra(grafo, 'BASE')
def custo_rota(rota):
    return sum(dict(grafo[a])[b] for a, b in zip(rota, rota[1:]))
rotas = []
for v in grafo:
    rb, rd = reconstruir(pais_bfs, 'BASE', v), reconstruir(pais_dij, 'BASE', v)
    rotas.append(dict(destino=v, bfs_rota=' → '.join(rb), bfs_trechos=trechos.get(v),
                      bfs_min=custo_rota(rb), dijkstra_rota=' → '.join(rd), dijkstra_min=dist.get(v)))
rotas = pd.DataFrame(rotas); display(rotas)
rotas.to_csv(OUT / 'rotas.csv', index=False)
diferencas = int((rotas.bfs_min > rotas.dijkstra_min + 1e-8).sum())
display(Markdown(f"Em **{diferencas} destinos**, a rota BFS tem maior tempo que a rota Dijkstra. "
    "BFS minimiza trechos, ignorando os pesos. Dijkstra minimiza a soma dos minutos, "
    "inclusive para deslocamentos entre dois atendimentos."))

,vertice,vizinhos_minutos
0,BASE,"[(VSO, 5), (PIN, 25)]"
1,VSO,"[(BASE, 5), (MBI, 3), (TVS, 4)]"
2,MBI,"[(VSO, 3), (BUT, 4)]"
3,BUT,"[(MBI, 4), (PIN, 3)]"
4,PIN,"[(BUT, 3), (FAL, 4), (BASE, 25)]"
5,FAL,"[(PIN, 4), (FRA, 3)]"
6,FRA,"[(FAL, 3), (FRE, 4)]"
7,FRE,"[(FRA, 4), (PTA, 3)]"
8,PTA,"[(FRE, 3), (HIG, 4)]"
9,HIG,"[(PTA, 4), (REP, 3)]"


Equipamentos individuais vinculados em toda a base: 201


,destino,bfs_rota,bfs_trechos,bfs_min,dijkstra_rota,dijkstra_min
0,BASE,BASE,0,0,BASE,0.0
1,VSO,BASE → VSO,1,5,BASE → VSO,5.0
2,MBI,BASE → VSO → MBI,2,8,BASE → VSO → MBI,8.0
3,BUT,BASE → PIN → BUT,2,28,BASE → VSO → MBI → BUT,12.0
4,PIN,BASE → PIN,1,25,BASE → VSO → MBI → BUT → PIN,15.0
5,FAL,BASE → PIN → FAL,2,29,BASE → VSO → MBI → BUT → PIN → FAL,19.0
6,FRA,BASE → PIN → FAL → FRA,3,32,BASE → VSO → MBI → BUT → PIN → FAL → FRA,22.0
7,FRE,BASE → PIN → FAL → FRA → FRE,4,36,BASE → VSO → MBI → BUT → PIN → FAL → FRA → FRE,26.0
8,PTA,BASE → PIN → FAL → FRA → FRE → PTA,5,39,BASE → VSO → MBI → BUT → PIN → FAL → FRA → FRE...,29.0
9,HIG,BASE → PIN → FAL → FRA → FRE → PTA → HIG,6,43,BASE → VSO → MBI → BUT → PIN → FAL → FRA → FRE...,33.0


Em **9 destinos**, a rota BFS tem maior tempo que a rota Dijkstra. BFS minimiza trechos, ignorando os pesos. Dijkstra minimiza a soma dos minutos, inclusive para deslocamentos entre dois atendimentos.

## 4. A* e comparação justa com Dijkstra
       h(v)=distância euclidiana esquemática(v,destino)/vmax, onde
       vmax=max(distância(u,v)/peso(u,v)) sobre as arestas. Assim h nunca excede
       o custo de uma rota, e h(u)≤peso(u,v)+h(v): heurística consistente.
       vmax é limite do modelo, sem interpretação como velocidade real.
       Ambos os algoritmos encerram ao retirar o destino da fila; compara-se o
       mesmo par origem/destino. O custo de preparar vmax está incluído no A*.

In [5]:
comparacao = []
for destino in grafo:
    dd, pp, ed = dijkstra(grafo, 'BASE', destino)
    ca, ra, ea = astar(grafo, coords, 'BASE', destino)
    assert math.isclose(ca, dd[destino])
    assert math.isclose(custo_rota(ra), ca)
    td = np.median(repeat(lambda: dijkstra(grafo, 'BASE', destino), number=100, repeat=5))*1e6/100
    ta = np.median(repeat(lambda: astar(grafo, coords, 'BASE', destino), number=100, repeat=5))*1e6/100
    comparacao.append(dict(destino=destino, custo_min=ca, rota_astar=' → '.join(ra),
        expansoes_dijkstra=ed, expansoes_astar=ea, dijkstra_us=td, astar_us=ta))
comparacao = pd.DataFrame(comparacao); display(comparacao)
comparacao.to_csv(OUT / 'comparacao_astar.csv', index=False)
display(Markdown(f"A* e Dijkstra devolveram custos iguais em todos os destinos. "
    f"Expansões totais: Dijkstra={comparacao.expansoes_dijkstra.sum()}, A*={comparacao.expansoes_astar.sum()}. "
    f"Medianas entre destinos: {comparacao.dijkstra_us.median():.2f} µs e {comparacao.astar_us.median():.2f} µs, respectivamente. "
    "Menos expansões não garantem menor tempo numa rede pequena, pois a heurística tem custo adicional. "
    "As coordenadas esquemáticas limitam a força da heurística."))

,destino,custo_min,rota_astar,expansoes_dijkstra,expansoes_astar,dijkstra_us,astar_us
0,BASE,0.0,BASE,1,1,0.318000,9.155003
1,VSO,5.0,BASE → VSO,2,2,0.880002,11.064000
2,MBI,8.0,BASE → VSO → MBI,3,3,1.547001,12.494996
3,BUT,12.0,BASE → VSO → MBI → BUT,5,4,2.520997,14.378000
4,PIN,15.0,BASE → VSO → MBI → BUT → PIN,6,5,2.886001,14.337003
5,FAL,19.0,BASE → VSO → MBI → BUT → PIN → FAL,7,6,3.474001,15.628999
6,FRA,22.0,BASE → VSO → MBI → BUT → PIN → FAL → FRA,8,7,4.123999,17.654998
7,FRE,26.0,BASE → VSO → MBI → BUT → PIN → FAL → FRA → FRE,9,8,4.806002,18.643001
8,PTA,29.0,BASE → VSO → MBI → BUT → PIN → FAL → FRA → FRE...,10,9,4.769000,19.032001
9,HIG,33.0,BASE → VSO → MBI → BUT → PIN → FAL → FRA → FRE...,11,10,5.918001,21.907999


A* e Dijkstra devolveram custos iguais em todos os destinos. Expansões totais: Dijkstra=91, A*=82. Medianas entre destinos: 3.47 µs e 15.63 µs, respectivamente. Menos expansões não garantem menor tempo numa rede pequena, pois a heurística tem custo adicional. As coordenadas esquemáticas limitam a força da heurística.

## 5. Plano de manutenção integrado
       `planejar_turno(base, horas)` usa a ordem do ranking, calcula o caminho do
       ponto atual ao próximo serviço e reserva tempo para voltar à base.
       Um item inviável é registrado e o próximo é tentado. Empates no ranking
       são resolvidos por identificador. É uma política gulosa de prioridade;
       não resolve o problema global de orienteering/roteamento ótimo.

In [6]:
planejar_turno = criar_planejador(grafo, ranking.dropna(subset=['criticidade']).to_dict('records'))
plano = planejar_turno('BASE', 8)
display(pd.DataFrame(plano['atendimentos']))
display({k: v for k, v in plano.items() if k != 'atendimentos'})
assert plano['total_min'] <= 480
(OUT / 'plano_turno.json').write_text(json.dumps(plano, ensure_ascii=False, indent=2), encoding='utf-8')
display(Markdown(f"O turno de 8 horas atende **{len(plano['atendimentos'])} escadas**, "
    f"consome **{plano['total_min']:.1f} min**, incluindo retorno, e deixa {plano['disponivel_min']:.1f} min livres. "
    "A duração de 45 min por serviço é uma hipótese; pausas, qualificações, peças e interdições não são modeladas. "
    "O plano apoia a discussão acadêmica e precisa de validação operacional."))

,escada_id,estacao,criticidade,saida_min,rota,deslocamento_min,servico_min,conclusao_min
0,SP04-6-ECR-PIN-ER21,PIN,138073.166667,0.0,"[BASE, VSO, MBI, BUT, PIN]",15.0,45.0,60.0
1,SP04-6-ECR-PIN-ER26,PIN,91417.883333,60.0,[PIN],0.0,45.0,105.0
2,SP04-6-ECR-PIN-ER15,PIN,30370.600000,105.0,[PIN],0.0,45.0,150.0
3,SP04-6-ECR-PTA-ER15,PTA,28487.533333,150.0,"[PIN, FAL, FRA, FRE, PTA]",14.0,45.0,209.0
4,SP04-6-ECR-PIN-ER22,PIN,23576.883333,209.0,"[PTA, FRE, FRA, FAL, PIN]",14.0,45.0,268.0
5,SP04-6-ECR-FRE-ER13,FRE,22870.150000,268.0,"[PIN, FAL, FRA, FRE]",11.0,45.0,324.0
6,SP04-6-ECR-FAL-ER09,FAL,18185.383333,324.0,"[FRE, FRA, FAL]",7.0,45.0,376.0
7,SP04-6-ECR-PTA-ER17,PTA,13512.966667,376.0,"[FAL, FRA, FRE, PTA]",10.0,45.0,431.0


{'nao_atendidas': ['SP04-6-ECR-PIN-ER20',
  'SP04-6-ECR-BUT-ER07',
  'SP04-6-ECR-LUZ-ER11',
  'SP04-6-ECR-PIN-ER16',
  'SP04-6-ECR-FAL-ER05',
  'SP04-6-ECR-FAL-ER06',
  'SP04-6-ECR-PIN-ER14',
  'SP04-6-ECR-BUT-ER13',
  'SP04-6-ECR-LUZ-ER07',
  'SP04-6-ECR-PIN-ER28',
  'SP04-6-ECR-PIN-ER31',
  'SP04-6-ECR-PIN-ER24',
  'SP04-6-ECR-LUZ-ER10',
  'SP04-6-ECR-PIN-ER02',
  'SP04-6-ECR-PIN-ER06',
  'SP04-6-ECR-HIG-ER09',
  'SP04-6-ECR-BUT-ER09',
  'SP04-6-ECR-HIG-ER07',
  'SP04-6-ECR-PIN-ER13',
  'SP04-6-ECR-MBI-ER08',
  'SP04-6-ECR-LUZ-ER12',
  'SP04-6-ECR-PIN-ER18',
  'SP04-6-ECR-PIN-ER04',
  'SP04-6-ECR-LUZ-ER02',
  'SP04-6-ECR-PIN-ER30',
  'SP04-6-ECR-PIN-ER09',
  'SP04-6-ECR-REP-ER59',
  'SP04-6-ECR-PIN-ER19',
  'SP04-6-ECR-PIN-ER32',
  'SP04-6-ECR-PIN-ER17',
  'SP04-6-ECR-HIG-ER24',
  'SP04-6-ECR-PIN-ER11',
  'SP04-6-ECR-BUT-ER06',
  'SP04-6-ECR-LUZ-ER19',
  'SP04-6-ECR-VSO-ER08',
  'SP04-6-ECR-LUZ-ER09',
  'SP04-6-ECR-LUZ-ER04',
  'SP04-6-ECR-LUZ-ER14',
  'SP04-6-ECR-PIN-ER01',
  'SP04-

O turno de 8 horas atende **8 escadas**, consome **460.0 min**, incluindo retorno, e deixa 20.0 min livres. A duração de 45 min por serviço é uma hipótese; pausas, qualificações, peças e interdições não são modeladas. O plano apoia a discussão acadêmica e precisa de validação operacional.

## 6. Sensibilidade e limites
       Compara o ranking original com uma versão que limita cada contribuição ao
       percentil 99 global, apenas como experimento; o ranking principal mantém
       os extremos. Também mostra o impacto da hipótese de duração de serviço.

In [7]:
teto = recorte.incremento.quantile(.99)
alternativo = recorte.assign(limitado=recorte.incremento.clip(upper=teto)).groupby('escada_id').limitado.sum(min_count=1)
top_original = set(ranking.dropna(subset=['criticidade']).head(10).escada_id)
top_alternativo = set(alternativo.nlargest(10).index)
print('Teto experimental P99:', teto, 'min')
print('Interseção dos top 10:', len(top_original & top_alternativo))
cenarios = []
for servico in [30, 45, 60]:
    r = ranking.dropna(subset=['criticidade']).copy(); r['servico_min'] = servico
    p = criar_planejador(grafo, r.to_dict('records'))('BASE', 8)
    cenarios.append(dict(servico_min=servico, atendimentos=len(p['atendimentos']), total_min=p['total_min']))
display(pd.DataFrame(cenarios))
display(Markdown(f"Os dois rankings compartilham {len(top_original & top_alternativo)} dos dez primeiros equipamentos. "
    "Variações mostram dependência em relação a eventos longos. Limitar durações não é uma limpeza justificada "
    "sem investigação. O ranking não mede degradação por sensores; a adaptação e os vínculos de estação "
    "devem ser avaliados pelo professor. Para a exigência literal, são necessários o dataset de sensores "
    "e uma função de incremento calibrada pela engenharia. "
    "O módulo sensores.py disponibiliza incrementos_sensores para essa integração futura: "
    "exige as seis colunas físicas, referências, escalas, pesos e sentidos fornecidos explicitamente. "
    "A integração temporal usa trapézios, rejeita valores ausentes e lacunas excessivas, "
    "e não aplica parâmetros inventados ao histórico de avarias."))

Teto experimental P99:

 1699.951333333328 min
Interseção dos top 10: 4


,servico_min,atendimentos,total_min
0,30,12,474.0
1,45,8,460.0
2,60,6,440.0


Os dois rankings compartilham 4 dos dez primeiros equipamentos. Variações mostram dependência em relação a eventos longos. Limitar durações não é uma limpeza justificada sem investigação. O ranking não mede degradação por sensores; a adaptação e os vínculos de estação devem ser avaliados pelo professor. Para a exigência literal, são necessários o dataset de sensores e uma função de incremento calibrada pela engenharia. O módulo sensores.py disponibiliza incrementos_sensores para essa integração futura: exige as seis colunas físicas, referências, escalas, pesos e sentidos fornecidos explicitamente. A integração temporal usa trapézios, rejeita valores ausentes e lacunas excessivas, e não aplica parâmetros inventados ao histórico de avarias.

# Complemento — SENSORES SIMULADOS

**DADOS ARTIFICIAIS, GERADOS PARA FINS DIDÁTICOS.** Este complemento foi solicitado
pelo grupo para demonstrar os algoritmos dos sinais ausentes na base real.
Não representa medições ou resultados da Motiva. As análises reais anteriores
permanecem independentes. Equipamentos fictícios têm prefixo `SIM-`.

Semente: 566903; 22 escadas; 96 instantes por escada; intervalo de 15 minutos.
As relações entre sinais foram inseridas no gerador. A base bruta contém duas
duplicatas exatas e três temperaturas ausentes intencionais. Somente essas lacunas
internas são interpoladas por equipamento, entre leituras igualmente espaçadas.
Extremos válidos são preservados. A aceitação da simulação depende do professor.

## S1. Incrementos de degradação didática
O módulo `sensores.py` calcula desvios normalizados: acima da referência para
vibração/temperatura/corrente, abaixo para óleo, bilateral para tensão/velocidade.
O índice é a média ponderada desses desvios; integra-se pelo método dos trapézios
em horas. D(0)=0, D(n)=D(n−1)+incremento(n). Pesos, referências e escalas constam
do JSON e são hipóteses, sem significado de limite técnico homologado.
Todos os equipamentos compartilham a mesma janela de 23h45min.

In [8]:
from simulacao import preparar as preparar_simulacao, SENSORES, UNIDADES
bruto_sim, sensores_sim, auditoria_sim = preparar_simulacao()
OUT_SIM = ROOT / 'resultados/simulacao'
OUT_SIM.mkdir(exist_ok=True)
print('DADOS SIMULADOS — SEM MEDIÇÕES REAIS DA MOTIVA')
display(pd.Series(auditoria_sim).to_frame('contagem'))
display(pd.DataFrame({'sensor': SENSORES, 'unidade_hipotetica': UNIDADES}))

from sensores import incrementos_sensores
config_sim=json.loads((ROOT/'dados/parametros_sensores_simulados.json').read_text())
display(pd.DataFrame(config_sim).T)
integrado_sim=incrementos_sensores(sensores_sim,config_sim,max_intervalo_horas=.25)
resultados_sim=[]; sequencias_sim={}
for escada,g in integrado_sim.groupby('escada_id'):
    valores=g.incremento.tolist(); sequencias_sim[escada]=valores
    a,b,c=recursiva(valores),memoizada(valores),tabulacao(valores)[-1]
    assert np.allclose([a,b],[c,c])
    resultados_sim.append(dict(escada_id=escada,estacao=g.estacao.iloc[0],criticidade=c,
        recursiva=a,memoizada=b,tabulacao=c,leituras=len(g),servico_min=45,origem='SIMULADO'))
ranking_sim=pd.DataFrame(resultados_sim).sort_values(['criticidade','escada_id'],ascending=[False,True])
display(ranking_sim)
ranking_sim.to_csv(OUT_SIM/'ranking_sensores_simulados.csv',index=False)
sensores_sim[['escada_id','estacao']].drop_duplicates().to_csv(ROOT/'dados/escadas_simuladas_estacoes.csv',index=False)
display(Markdown(f"As três versões devolveram os mesmos índices para {len(ranking_sim)} escadas fictícias. "
    f"A maior exposição simulada é de **{ranking_sim.iloc[0].escada_id}**, com {ranking_sim.iloc[0].criticidade:.4f} "
    "unidades de índice-hora. Os resultados refletem os parâmetros escolhidos e não medem dano físico validado."))

DADOS SIMULADOS — SEM MEDIÇÕES REAIS DA MOTIVA


,contagem
duplicatas,2
valores_interpolados,3


,sensor,unidade_hipotetica
0,vibracao,mm/s (hipótese)
1,temperatura,°C
2,corrente,A
3,tensao,V
4,nivel_oleo,% (hipótese)
5,velocidade,m/s


,referencia,escala,peso,sentido
vibracao,2,1,2,alto
temperatura,50,10,1,alto
corrente,11,4,1,alto
tensao,380,10,1,bilateral
nivel_oleo,80,20,1,baixo
velocidade,0.5,0.05,1,bilateral


,escada_id,estacao,criticidade,recursiva,memoizada,tabulacao,leituras,servico_min,origem
11,SIM-LUZ-ER02,LUZ,7.081375,7.081375,7.081375,7.081375,96,45,SIMULADO
19,SIM-REP-ER02,REP,6.081335,6.081335,6.081335,6.081335,96,45,SIMULADO
10,SIM-LUZ-ER01,LUZ,6.053640,6.053640,6.053640,6.053640,96,45,SIMULADO
9,SIM-HIG-ER02,HIG,5.542683,5.542683,5.542683,5.542683,96,45,SIMULADO
18,SIM-REP-ER01,REP,5.224847,5.224847,5.224847,5.224847,96,45,SIMULADO
17,SIM-PTA-ER02,PTA,4.297588,4.297588,4.297588,4.297588,96,45,SIMULADO
8,SIM-HIG-ER01,HIG,4.283775,4.283775,4.283775,4.283775,96,45,SIMULADO
16,SIM-PTA-ER01,PTA,3.495172,3.495172,3.495172,3.495172,96,45,SIMULADO
7,SIM-FRE-ER02,FRE,3.347751,3.347751,3.347751,3.347751,96,45,SIMULADO
6,SIM-FRE-ER01,FRE,2.803161,2.803161,2.803161,2.803161,96,45,SIMULADO


As três versões devolveram os mesmos índices para 22 escadas fictícias. A maior exposição simulada é de **SIM-LUZ-ER02**, com 7.0814 unidades de índice-hora. Os resultados refletem os parâmetros escolhidos e não medem dano físico validado.

## S2. Medição das três versões com sensores simulados

In [9]:
amostra_sim=max(sequencias_sim.values(),key=len)
tempos_sim=[]
for n in [16,48,96]:
    for nome,func in [('recursiva',recursiva),('memoizada',memoizada),('tabulacao',lambda v:tabulacao(v)[-1])]:
        med=np.median(repeat(lambda:func(amostra_sim[:n]),number=100,repeat=7))*1e6/100
        tempos_sim.append(dict(n=n,versao=nome,mediana_us=med))
tempos_sim=pd.DataFrame(tempos_sim); display(tempos_sim)
tempos_sim.to_csv(OUT_SIM/'desempenho_sensores_simulados.csv',index=False)
melhor_sim=tempos_sim[tempos_sim.n.eq(96)].sort_values('mediana_us').iloc[0]
display(Markdown(f"Para 96 leituras, a menor mediana foi de {melhor_sim.versao}: {melhor_sim.mediana_us:.2f} µs. "
    "Todas as versões são lineares para uma consulta; memoização não oferece ganho exponencial nesta recorrência. "
    "O cache começa vazio em cada chamada. Tempos incluem a seleção do prefixo e variam conforme o ambiente."))

,n,versao,mediana_us
0,16,recursiva,2.020998
1,16,memoizada,3.706999
2,16,tabulacao,1.075999
3,48,recursiva,5.802000
4,48,memoizada,11.322997
5,48,tabulacao,3.068997
6,96,recursiva,12.557004
7,96,memoizada,23.415000
8,96,tabulacao,6.255000


Para 96 leituras, a menor mediana foi de tabulacao: 6.25 µs. Todas as versões são lineares para uma consulta; memoização não oferece ganho exponencial nesta recorrência. O cache começa vazio em cada chamada. Tempos incluem a seleção do prefixo e variam conforme o ambiente.

## S3. Ranking simulado integrado à rede e ao turno
A mesma rede didática de BFS, Dijkstra e A* das seções anteriores é usada aqui.
Os equipamentos `SIM-` não substituem ou alteram o ranking de avarias reais.

In [10]:
assert set(ranking_sim.estacao)<=set(grafo)
rotas_sim=[]
for estacao in sorted(set(ranking_sim.estacao)):
    db,pb=bfs(grafo,'BASE'); dd,pdij,ed=dijkstra(grafo,'BASE',estacao)
    ca,ra,ea=astar(grafo,coords,'BASE',estacao)
    assert math.isclose(ca,dd[estacao])
    rotas_sim.append(dict(estacao=estacao,bfs_rota=' → '.join(reconstruir(pb,'BASE',estacao)),
        bfs_trechos=db[estacao],dijkstra_min=dd[estacao],astar_min=ca,astar_rota=' → '.join(ra)))
display(pd.DataFrame(rotas_sim))
planejar_turno_simulado=criar_planejador(grafo,ranking_sim.to_dict('records'))
plano_sim=planejar_turno_simulado('BASE',8)
assert plano_sim['total_min']<=480
display(pd.DataFrame(plano_sim['atendimentos']))
display({k:v for k,v in plano_sim.items() if k!='atendimentos'})
(OUT_SIM/'plano_sensores_simulados.json').write_text(json.dumps(plano_sim,ensure_ascii=False,indent=2),encoding='utf-8')
display(Markdown(f"O plano baseado exclusivamente nos sensores artificiais atende {len(plano_sim['atendimentos'])} "
    f"escadas em {plano_sim['total_min']:.1f} minutos, com retorno à base. "
    "O indicador, os serviços de 45 min e os deslocamentos são hipóteses. "
    "A demonstração cobre o fluxo sensores → índice → ranking → rotas → turno, "
    "mas não comprova criticidade real nem substitui o dataset de sensores solicitado à Motiva."))

,estacao,bfs_rota,bfs_trechos,dijkstra_min,astar_min,astar_rota
0,BUT,BASE → PIN → BUT,2,12.0,12.0,BASE → VSO → MBI → BUT
1,FAL,BASE → PIN → FAL,2,19.0,19.0,BASE → VSO → MBI → BUT → PIN → FAL
2,FRA,BASE → PIN → FAL → FRA,3,22.0,22.0,BASE → VSO → MBI → BUT → PIN → FAL → FRA
3,FRE,BASE → PIN → FAL → FRA → FRE,4,26.0,26.0,BASE → VSO → MBI → BUT → PIN → FAL → FRA → FRE
4,HIG,BASE → PIN → FAL → FRA → FRE → PTA → HIG,6,33.0,33.0,BASE → VSO → MBI → BUT → PIN → FAL → FRA → FRE...
5,LUZ,BASE → PIN → FAL → FRA → FRE → PTA → HIG → REP...,8,40.0,40.0,BASE → VSO → MBI → BUT → PIN → FAL → FRA → FRE...
6,MBI,BASE → VSO → MBI,2,8.0,8.0,BASE → VSO → MBI
7,PIN,BASE → PIN,1,15.0,15.0,BASE → VSO → MBI → BUT → PIN
8,PTA,BASE → PIN → FAL → FRA → FRE → PTA,5,29.0,29.0,BASE → VSO → MBI → BUT → PIN → FAL → FRA → FRE...
9,REP,BASE → PIN → FAL → FRA → FRE → PTA → HIG → REP,7,36.0,36.0,BASE → VSO → MBI → BUT → PIN → FAL → FRA → FRE...


,escada_id,estacao,criticidade,saida_min,rota,deslocamento_min,servico_min,conclusao_min
0,SIM-LUZ-ER02,LUZ,7.081375,0.0,"[BASE, VSO, MBI, BUT, PIN, FAL, FRA, FRE, PTA,...",40.0,45.0,85.0
1,SIM-REP-ER02,REP,6.081335,85.0,"[LUZ, REP]",4.0,45.0,134.0
2,SIM-LUZ-ER01,LUZ,6.053640,134.0,"[REP, LUZ]",4.0,45.0,183.0
3,SIM-HIG-ER02,HIG,5.542683,183.0,"[LUZ, REP, HIG]",7.0,45.0,235.0
4,SIM-REP-ER01,REP,5.224847,235.0,"[HIG, REP]",3.0,45.0,283.0
5,SIM-PTA-ER02,PTA,4.297588,283.0,"[REP, HIG, PTA]",7.0,45.0,335.0
6,SIM-HIG-ER01,HIG,4.283775,335.0,"[PTA, HIG]",4.0,45.0,384.0
7,SIM-PTA-ER01,PTA,3.495172,384.0,"[HIG, PTA]",4.0,45.0,433.0


{'nao_atendidas': ['SIM-FRE-ER02',
  'SIM-FRE-ER01',
  'SIM-FRA-ER02',
  'SIM-FRA-ER01',
  'SIM-FAL-ER02',
  'SIM-FAL-ER01',
  'SIM-PIN-ER02',
  'SIM-BUT-ER02',
  'SIM-PIN-ER01',
  'SIM-MBI-ER02',
  'SIM-BUT-ER01',
  'SIM-VSO-ER02',
  'SIM-MBI-ER01',
  'SIM-VSO-ER01'],
 'rota_retorno': ['PTA',
  'FRE',
  'FRA',
  'FAL',
  'PIN',
  'BUT',
  'MBI',
  'VSO',
  'BASE'],
 'retorno_min': 29.0,
 'total_min': 462.0,
 'disponivel_min': 18.0}

O plano baseado exclusivamente nos sensores artificiais atende 8 escadas em 462.0 minutos, com retorno à base. O indicador, os serviços de 45 min e os deslocamentos são hipóteses. A demonstração cobre o fluxo sensores → índice → ranking → rotas → turno, mas não comprova criticidade real nem substitui o dataset de sensores solicitado à Motiva.